In [ ]:
pip install transformers

In [ ]:
import pandas as pd
import numpy as np
import os
from transformers import BertConfig, BertTokenizer
from transformers import BertModel
from transformers import get_linear_schedule_with_warmup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset,DataLoader,RandomSampler
from transformers import BertForSequenceClassification, AdamW, BertConfig
from transformers import get_linear_schedule_with_warmup
import pickle
from sklearn.metrics import classification_report
import seaborn as sns
import re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
torch.cuda.empty_cache()

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f'device: {device}')

train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')

In [ ]:
x_train_df = pd.read_csv('/content/drive/MyDrive/datas/x_train.csv')
x_test_df = pd.read_csv('/content/drive/MyDrive/datas/x_test.csv')
x_dev_df = pd.read_csv('/content/drive/MyDrive/datas/x_dev.csv')
y_train_df = pd.read_csv('/content/drive/MyDrive/datas/y_train.csv')
y_test_df = pd.read_csv('/content/drive/MyDrive/datas/y_test.csv')
y_dev_df = pd.read_csv('/content/drive/MyDrive/datas/y_dev.csv')

In [ ]:
x_train = x_train_df.doc.values
x_test = x_test_df.doc.values
x_dev = x_dev_df.doc.values
y_train = y_train_df.label.values
y_test = y_test_df.label.values
y_dev = y_dev_df.label.values

In [ ]:
max_len = 512
train_batch_size = 16
dev_batch_size = 16
test_batch_size = 16

epochs = 3



MODEL_NAME = 'HooshvareLab/bert-fa-base-uncased-clf-persiannews'
# OUTPUT_PATH = '/content/bert/trained_model.bin'

# os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

In [ ]:
label2id = {'Economic': 1, 'International': 0, 'Political': 5,'Science Technology': 3,'Cultural Art': 4,'Sport':6,"Social":2}
id2label = {0:'International',1:'Economic',2:'Social',3:'Science Technology',4:'Cultural Art',5:'Political',6:'Sport'}

In [ ]:
config = BertConfig.from_pretrained(MODEL_NAME, **{ 'label2id': label2id,'id2label': id2label})

In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
x_train_df['doc'][:5]

In [ ]:
dic={}
for i in range(len(x_train)):
  if (len(str(x_train[i]))) not in dic:
     dic[len(str(x_train[i]))]=1
  else:
    dic[len(str(x_train[i]))]+=1

for i in range(len(x_test)):
  if (len(str(x_test[i]))) not in dic:
     dic[len(str(x_test[i]))]=1
  else:
    dic[len(str(x_test[i]))]+=1

for i in range(len(x_dev)):
  if (len(str(x_dev[i]))) not in dic:
     dic[len(str(x_dev[i]))]=1
  else:
    dic[len(str(x_dev[i]))]+=1


In [ ]:
{k: v for k, v in sorted(dic.items(), key=lambda item: item[1],reverse=True)}

In [ ]:
print(x_train[0])
print(tokenizer.tokenize(x_train[0]))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(x_train[0])))

In [ ]:
encoded_x_train=[]
mask_x_train=[]
for sent in x_train:
  encoded_dic = tokenizer.encode_plus(sent, add_special_tokens = True, max_length = max_len, pad_to_max_length = True,truncation=True, return_attention_mask = True, return_tensors = 'pt')
  encoded_x_train.append(encoded_dic['input_ids'])
  mask_x_train.append(encoded_dic['attention_mask'])

encoded_x_train = torch.cat(encoded_x_train, dim=0)
mask_x_train = torch.cat(mask_x_train, dim=0)
y_train = torch.tensor(y_train)

In [ ]:
encoded_x_test=[]
mask_x_test=[]
for sent in x_test:
  encoded_dic = tokenizer.encode_plus(sent, add_special_tokens = True, max_length = max_len,truncation=True, pad_to_max_length = True, return_attention_mask = True, return_tensors = 'pt')
  encoded_x_test.append(encoded_dic['input_ids'])
  mask_x_test.append(encoded_dic['attention_mask'])

encoded_x_test = torch.cat(encoded_x_test, dim=0)
mask_x_test = torch.cat(mask_x_test, dim=0)
y_test = torch.tensor(y_test)

In [ ]:
encoded_x_dev=[]
mask_x_dev=[]
for sent in x_dev:
  encoded_dic = tokenizer.encode_plus(sent, add_special_tokens = True, max_length = max_len, pad_to_max_length = True, return_attention_mask = True, return_tensors = 'pt')
  encoded_x_dev.append(encoded_dic['input_ids'])
  mask_x_dev.append(encoded_dic['attention_mask'])

encoded_x_dev = torch.cat(encoded_x_dev, dim=0)
mask_x_dev = torch.cat(mask_x_dev, dim=0)
y_dev = torch.tensor(y_dev)

In [ ]:
print(encoded_x_train.shape)
print(encoded_x_test.shape)
print(encoded_x_dev.shape)
print(y_train.shape)
print(y_test.shape)
print(y_dev.shape)

In [ ]:
train_dataset = TensorDataset(encoded_x_train, mask_x_train, y_train)
test_dataset = TensorDataset(encoded_x_test, mask_x_test, y_test)
dev_dataset = TensorDataset(encoded_x_dev, mask_x_dev, y_dev)

In [ ]:
train_batch = DataLoader(train_dataset, sampler = RandomSampler(train_dataset), batch_size = train_batch_size)
test_batch = DataLoader(test_dataset, sampler = RandomSampler(test_dataset), batch_size = test_batch_size)
dev_batch = DataLoader(dev_dataset, sampler = RandomSampler(dev_dataset), batch_size = dev_batch_size)

In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME,config=config,ignore_mismatched_sizes=True)
model.cuda()

In [ ]:
optimizer = AdamW(model.parameters(),lr = 2e-5)

In [ ]:
total_steps = len(train_batch) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer,num_warmup_steps=0, num_training_steps = total_steps)

In [ ]:
def accuracy(preds, labels):
    pred = np.argmax(preds, axis=1).flatten()
    labels = labels.flatten()
    return np.sum(pred == labels) / len(labels)

In [ ]:


training_stats = []

for epoch_i in range(0, epochs):


    print(' Epoch {:} / {:} '.format(epoch_i + 1, epochs))
    total_train_loss = 0
    model.train()
    for step, batch in enumerate(train_batch):

        encoded_batch_x_train = batch[0].to(device)
        mask_batch_x_train = batch[1].to(device)
        labels_batch_x_train = batch[2].to(device)
        model.zero_grad()
        result = model(encoded_batch_x_train,
                       token_type_ids=None,
                       attention_mask=mask_batch_x_train,
                       labels=labels_batch_x_train,
                       return_dict=True)

        loss = result.loss
        logits = result.logits
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_batch)
    print("  Average training loss: {0:.2f}".format(avg_train_loss))

    print("Running Validation...")

    model.eval()
    total_eval_accuracy = 0
    total_eval_loss = 0
    nb_eval_steps = 0

    for batch in dev_batch:
        encoded_batch_x_dev = batch[0].to(device)
        mask_batch_x_dev = batch[1].to(device)
        labels_batch_x_dev = batch[2].to(device)
        with torch.no_grad():
            result = model(encoded_batch_x_dev,
                           token_type_ids=None,
                           attention_mask=mask_batch_x_dev,
                           labels=labels_batch_x_dev,
                           return_dict=True)

        loss = result.loss
        logits = result.logits
        total_eval_loss += loss.item()
        logits = logits.detach().cpu().numpy()
        label_ids = labels_batch_x_dev.to('cpu').numpy()


        total_eval_accuracy += accuracy(logits, label_ids)


    avg_val_accuracy = total_eval_accuracy / len(dev_batch)
    print("  Accuracy: {0:.2f}".format(avg_val_accuracy))

    avg_val_loss = total_eval_loss / len(dev_batch)

    print("  Validation Loss: {0:.2f}".format(avg_val_loss))

    training_stats.append(
        {
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'Valid. Accur.': avg_val_accuracy
        }
    )

print("")
print("Training complete!")

In [ ]:
filename = 'bert_model.pt'
torch.save(model, filename)

In [ ]:
df_stats = pd.DataFrame(data=training_stats)
df_stats = df_stats.set_index('epoch')
df_stats

In [ ]:
import matplotlib.pyplot as plt
% matplotlib inline
sns.set(style='darkgrid')
sns.set(font_scale=1.5)
plt.rcParams["figure.figsize"] = (12,6)
plt.plot(df_stats['Training Loss'], 'b-o', label="Training")
plt.plot(df_stats['Valid. Loss'], 'g-o', label="Validation")
plt.title("Training & Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.xticks([1, 2, 3])

plt.show()

In [ ]:
model.eval()
predictions , true_labels = [], []
for batch in test_batch:

  batch = tuple(t.to(device) for t in batch)
  encoded_batch_x_test, mask_batch_x_test, labels_batch_x_test = batch
  with torch.no_grad():
      result = model(encoded_batch_x_test,
                     token_type_ids=None,
                     attention_mask=mask_batch_x_test,
                     return_dict=True)

  logits = result.logits

  logits = logits.detach().cpu().numpy()
  label_ids =labels_batch_x_test .to('cpu').numpy()
  predictions.append(logits)
  true_labels.append(label_ids)

print('    DONE.')

In [ ]:
np.argmax(predictions[0], axis=1)

In [ ]:
true_labels[1]

In [ ]:
predicts = []
t_labels = []
for i in range(len(true_labels)):
    t_labels.extend(true_labels[i])
    predict_label = np.argmax(predictions[i], axis=1).tolist()
    predicts.extend(predict_label)

In [ ]:
target_name = ['beinolmelal', 'eghtesadi', 'ejtemaee', 'elmivadaneshghai', 'farhanghonarvaresane', 'siasi', 'varzeshi']
print(classification_report(t_labels, predicts, target_names = target_name))

In [ ]:
bert_model=torch.load('/content/drive/MyDrive/model/bert_model.pt')

In [ ]:
dicconvert = {
            'ك': 'ک',
            'دِ': 'د',
            'بِ': 'ب',
            'زِ': 'ز',
            'ذِ': 'ذ',
            'شِ': 'ش',
            'سِ': 'س',
            'ى': 'ی',
            'ي': 'ی',
            '١': '۱',
            '٢': '۲',
            '٣': '۳',
            '٤': '۴',
            '٥': '۵',
            '٦': '۶',
            '٧': '۷',
            '٨': '۸',
            '٩': '۹',
            '٠': '۰',
        }
numbers = "۰۱۲۳۴۵۶۷۸۹" + "0123456789"



In [ ]:
doc="""
با توجه به اینکه بی حجابی در جامعه اسلامی به اوج خود رسیده است و هنگامی که به این افراد در مورد پوشش بدشان تذکر داده می شود اکثرا فحاشی کرده و در مواردی که مشاهده شده آمرین به معروف را مورد حمله فیزیکی قرار می دهند درخواست داریم در مکان‌های دولتی به خانم‌های بدحجاب و آقایان با پوششهای زننده خدماتی ارائه نشود.
"""

In [ ]:
sentence=[]
contents = ""

for word in doc.split():

    word = "".join([w if w not in dicconvert else dicconvert[w] for w in word])
    word = "".join([w for w in word if w not in numbers])
    word = re.sub(r'[^\w\s]', '', word)
    word = re.sub("[a-zA-Z]+", "", word)
    word = word.strip()
    if (word != "") and (len(word) > 1):
        sentence.append(word)

contents=" ".join(sentence)

In [ ]:
contents

In [ ]:
encoded_test_doc=[]
mask_test_doc=[]

encoded_dic = tokenizer.encode_plus(contents, add_special_tokens = True, max_length = max_len,truncation=True, pad_to_max_length = True, return_attention_mask = True, return_tensors = 'pt')
encoded_test_doc.append(encoded_dic['input_ids'])
mask_test_doc.append(encoded_dic['attention_mask'])

encoded_test_doc = torch.cat(encoded_test_doc, dim=0)
mask_test_doc = torch.cat(mask_test_doc, dim=0)


In [ ]:
encoded_test_doc

In [ ]:
bert_model.eval()
predictions = []

encoded_doc = encoded_test_doc.to(device)
mask_doc = mask_test_doc.to(device)
with torch.no_grad():
  result = bert_model(encoded_doc,
                     token_type_ids=None,
                     attention_mask=mask_doc,
                     return_dict=True)

  logits = result.logits

  logits = logits.detach().cpu().numpy()
  predictions.append(logits)

print('    DONE.')

In [ ]:
predict_label_doc = np.argmax(predictions[0], axis=1).tolist()

In [ ]:
predict_label_doc